# Variantes de KNN para Classificação Binária com Desequilíbrio de Classes

**Aprendizagem Automática I (CC2008) — Universidade do Porto, 2025/2026**

---

### Índice

1. [Configuração e Importações](#1)
2. [Carregamento e Exploração dos Dados](#2)
3. [O Algoritmo KNN — Implementação de Raiz](#3)
4. [Porque é que o KNN falha com desequilíbrio de classes](#4)
5. [Proposta: KNNFairRank — Correção por Estatísticas de Ordem](#5)
6. [Estudo Empírico — Configuração Experimental](#6)
7. [Resultados e Análise](#7)
8. [Testes Estatísticos](#8)
9. [Discussão](#9)
10. [Conclusões e Trabalho Futuro](#10)

---
<a id='1'></a>
## 1. Configuração e Importações

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Markdown
from scipy import stats as sp_stats

from src.utils.config import load_config, get_project_root
from src.data.loader import load_all_datasets
from src.data.preprocessing import binarise_labels, standardise, remove_constant_features
from src.algorithms.knn_base import KNNClassifier, KNNClassifierFast

cfg  = load_config()
SEED = cfg["random_seed"]
ROOT = get_project_root()
TAB_DIR = ROOT / cfg["paths"]["results_tables"]
FIG_DIR = ROOT / cfg["paths"]["results_figures"]
TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150})

print(f"Raiz do projeto: {ROOT}")
print(f"Semente aleatória: {SEED}")

---
<a id='2'></a>
## 2. Carregamento e Exploração dos Dados

Utilizamos o conjunto de *benchmarks* para **desequilíbrio de classes em classificação binária** (Dataset Group 2 do enunciado). Os datasets são carregados automaticamente a partir da pasta `class_imbalance/`, com deteção automática da coluna-alvo, binarização de *features* categóricas (one-hot encoding) e imputação de valores em falta pela mediana.

In [ ]:
datasets = load_all_datasets()
print(f"Datasets carregados: {len(datasets)}")

In [ ]:
# Tabela-resumo dos datasets
rows = []
for ds in datasets:
    y = binarise_labels(ds.y)
    counts = np.bincount(y)
    ir = counts[0] / counts[1] if counts[1] > 0 else float('inf')
    rows.append({
        "Dataset": ds.name, "N amostras": len(y), "N features": ds.X.shape[1],
        "N maioria": counts[0], "N minoria": counts[1], "IR": round(ir, 2)
    })

summary_df = pd.DataFrame(rows).sort_values("IR").reset_index(drop=True)
summary_df.to_csv(TAB_DIR / "dataset_summary.csv", index=False)

display(Markdown("### Resumo dos Datasets de Benchmark"))
display(Markdown(
    f"- **{len(datasets)}** datasets binários\n"
    f"- IR (rácio de desequilíbrio) varia de **{summary_df['IR'].min():.1f}:1** a **{summary_df['IR'].max():.1f}:1**\n"
    f"- Nº de amostras: {summary_df['N amostras'].min()} a {summary_df['N amostras'].max()}\n"
    f"- Nº de features: {summary_df['N features'].min()} a {summary_df['N features'].max()}"
))
display(summary_df.head(15))

In [ ]:
# Distribuição dos rácios de desequilíbrio
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(summary_df["IR"], bins=20, edgecolor="white", color="steelblue")
axes[0].set(xlabel="Rácio de Desequilíbrio (IR = N_maj / N_min)",
            ylabel="Contagem",
            title="Distribuição do IR na suite de benchmark")

axes[1].hist(summary_df["N amostras"], bins=20, edgecolor="white", color="coral")
axes[1].set(xlabel="Número de Amostras",
            ylabel="Contagem",
            title="Distribuição do tamanho dos datasets")

plt.tight_layout()
plt.savefig(FIG_DIR / "distribuicao_datasets.png", dpi=150, bbox_inches="tight")
plt.show()

---
<a id='3'></a>
## 3. O Algoritmo KNN — Implementação de Raiz

### 3.1 Descrição do Algoritmo

O **K-Nearest Neighbours (KNN)** é um classificador não-paramétrico que classifica um ponto $x$ por votação maioritária entre os seus $k$ vizinhos mais próximos no conjunto de treino:

$$\hat{y}(x) = \arg\max_{c \in C} \sum_{i=1}^{k} \mathbb{1}[y_{(i)} = c]$$

onde $(i)$ denota o $i$-ésimo vizinho mais próximo de $x$ segundo a distância Euclidiana.

### 3.2 Implementação

A implementação base foi adaptada do repositório [rushter/MLAlgorithms](https://github.com/rushter/MLAlgorithms/blob/master/mla/knn.py), com as seguintes modificações:

| Classe | Descrição |
|---|---|
| `KNNClassifier` | KNN padrão implementado de raiz (sem sklearn) |
| `KNNClassifierFast` | Versão vectorizada via `scipy.cdist` — resultados idênticos, execução mais rápida |
| `KNNOptK` | Seleção automática de $k$ via cross-validation interna |

Todas as classes seguem a interface do sklearn (`fit`/`predict`/`predict_proba`), o que permite integração direta no pipeline de benchmarking.

### 3.3 KNNClassifierFast — Optimização Vectorizada

A implementação original do rushter usa um loop Python que chama `scipy.euclidean` uma vez por ponto de treino. `KNNClassifierFast` substitui este loop por uma única chamada a `scipy.cdist`, que calcula todas as distâncias numa única operação C/BLAS. As previsões são **bit-for-bit idênticas** — apenas a velocidade muda.

In [ ]:
# Demonstração: KNNClassifier vs KNNClassifierFast — tempo de execução
import time
from src.algorithms.knn_base import KNNClassifier, KNNClassifierFast

timing_ds = max((ds for ds in datasets if len(ds.X) >= 1000), key=lambda ds: len(ds.X))
X_t = remove_constant_features(timing_ds.X)
y_t = binarise_labels(timing_ds.y)
n_train = min(2000, int(0.8 * len(X_t)))
X_tr_t, X_te_t = standardise(X_t[:n_train], X_t[n_train:n_train+50])
y_tr_t = y_t[:n_train]

print(f"Dataset: {timing_ds.name} ({n_train} treino, 50 teste, {X_t.shape[1]} features)\n")

tempos = {}
for nome, clf in [("KNNClassifier (original)", KNNClassifier(k=5)),
                   ("KNNClassifierFast (vectorizado)", KNNClassifierFast(k=5))]:
    clf.fit(X_tr_t, y_tr_t)
    t0 = time.perf_counter()
    preds = clf.predict(X_te_t)
    dt = time.perf_counter() - t0
    tempos[nome] = dt
    print(f"{nome}: {dt:.3f}s")

speedup = tempos["KNNClassifier (original)"] / tempos["KNNClassifierFast (vectorizado)"]
print(f"\nAceleração: {speedup:.1f}×")

### 3.4 KNNOptK — Seleção Automática de $k$

Em vez de usar um $k$ fixo, `KNNOptK` seleciona o melhor $k$ via **cross-validation interna estratificada** (3-fold). O espaço de busca são todos os valores ímpares de 1 a $\lfloor\sqrt{n_{\text{treino}}}\rfloor$, avaliados por *balanced accuracy*. Esta é a nossa **baseline justa da Fase 1**: qualquer melhoria dos algoritmos da Fase 2 reflete o valor da adaptação proposta, e não uma melhor escolha de $k$.

In [ ]:
# Distribuição dos k selecionados pelo KNNOptK
_optk_cache = TAB_DIR / "opt_k_raw.csv"
if _optk_cache.exists():
    opt_k_df = pd.read_csv(_optk_cache)
    print(f"Resultados KNNOptK carregados da cache ({len(opt_k_df)} linhas).")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(opt_k_df["best_k"], bins=20, edgecolor="white", color="steelblue")
    axes[0].set(xlabel="k selecionado", ylabel="Contagem",
                title="Distribuição do k ótimo (por fold)")
    
    per_ds_optk = opt_k_df.groupby("dataset")[["IR", "best_k"]].mean()
    axes[1].scatter(per_ds_optk["IR"], per_ds_optk["best_k"], alpha=0.6, edgecolors="k", s=40)
    axes[1].set(xlabel="Rácio de Desequilíbrio (IR)", ylabel="k médio selecionado",
                title="k ótimo vs. grau de desequilíbrio")
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / "optk_distribuicao.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    pct_k1 = (opt_k_df["best_k"] == 1).mean() * 100
    print(f"\nk=1 selecionado em {pct_k1:.0f}% dos folds — indica que vizinhanças maiores diluem a minoria.")
else:
    print("Cache opt_k_raw.csv não encontrada. Execute o notebook phase1_baseline.ipynb primeiro.")

---
<a id='4'></a>
## 4. Porque é que o KNN Falha com Desequilíbrio de Classes

### 4.1 O Viés de Votação

Quando as classes são desequilibradas ($N_{\text{maj}} \gg N_{\text{min}}$), a vizinhança de $k$ pontos é **dominada pela classe maioritária** simplesmente porque existem mais pontos dessa classe. Isto desloca a fronteira de decisão na direção da classe minoritária, prejudicando sistematicamente a sua deteção.

### 4.2 O Viés Estrutural — Estatísticas de Distância

O problema é mais profundo do que a simples contagem. Mesmo com $k=1$, a classe maioritária tem uma **vantagem estatística**: com mais pontos no espaço, a distância esperada ao vizinho mais próximo maioritário é **sistematicamente menor** do que ao vizinho mais próximo minoritário.

Formalmente, assumindo uma distribuição Poisson uniforme com $N_c$ pontos da classe $c$ num espaço de dimensão $d$:

$$\mathbb{E}[d_1^{\text{min}}] / \mathbb{E}[d_1^{\text{maj}}] = r^{1/d}$$

onde $r = N_{\text{maj}} / N_{\text{min}}$ é o rácio de desequilíbrio. Isto significa que, **mesmo quando as distribuições das duas classes são idênticas**, o KNN favorece a maioria.

In [ ]:
# Demonstração empírica: rácio de distâncias ao rank-1 vs IR

def distancia_esperada_rank1(ir_values, d=2, n_maj=200, n_trials=200):
    """Rácio empírico E[d1_min] / E[d1_maj] em função do IR num dataset Poisson 2D."""
    ratios = []
    for ir in ir_values:
        n_min = max(1, int(n_maj / ir))
        rng = np.random.default_rng(42)
        diffs = []
        for _ in range(n_trials):
            X_maj = rng.uniform(0, 1, (n_maj, d))
            X_min = rng.uniform(0, 1, (n_min, d))
            q = rng.uniform(0, 1, (1, d))
            d_maj = np.min(np.sqrt(((X_maj - q) ** 2).sum(1)))
            d_min = np.min(np.sqrt(((X_min - q) ** 2).sum(1)))
            diffs.append(d_min / max(d_maj, 1e-12))
        ratios.append(np.mean(diffs))
    return np.array(ratios)

ir_vals = np.arange(1, 31)
ratios_2d = distancia_esperada_rank1(ir_vals, d=2)
ratios_10d = distancia_esperada_rank1(ir_vals, d=10)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ir_vals, ratios_2d, 'o-', label='d=2', markersize=4)
ax.plot(ir_vals, ratios_10d, 's-', label='d=10', markersize=4)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Comparação justa')
ax.set(xlabel="Rácio de Desequilíbrio (IR)",
       ylabel="E[d₁ᵐⁱⁿ] / E[d₁ᵐᵃʲ]",
       title="Viés estrutural: a distância ao vizinho minoritário infla com o IR")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "vies_estrutural.png", dpi=150, bbox_inches="tight")
plt.show()

print("Quando o rácio > 1.0, o vizinho minoritário está, em média, mais longe —")
print("o KNN vai preferir a maioria mesmo que as distribuições sejam idênticas.")

In [ ]:
# Visualização 2D: deslocamento da fronteira de decisão

def criar_dataset_2d(n_maj, n_min, seed=42):
    rng_l = np.random.default_rng(seed)
    X_maj = rng_l.multivariate_normal([0, 0], [[1, 0], [0, 1]], n_maj)
    X_min = rng_l.multivariate_normal([1.5, 1.5], [[0.6, 0], [0, 0.6]], n_min)
    return np.vstack([X_maj, X_min]), np.array([0]*n_maj + [1]*n_min)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (n_maj, n_min, titulo) in zip(axes, [(100, 100, 'IR = 1:1'),
                                               (200, 20, 'IR = 10:1'),
                                               (300, 10, 'IR = 30:1')]):
    X_s, y_s = criar_dataset_2d(n_maj, n_min)
    from src.algorithms.knn_base import KNNClassifierFast
    clf_s = KNNClassifierFast(k=5).fit(X_s, y_s)
    
    # Grelha de decisão
    xx, yy = np.meshgrid(np.linspace(-3, 4, 150), np.linspace(-3, 4, 150))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = clf_s.predict(grid).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.15, levels=[0, 0.5, 1], colors=['#3b82f6', '#ef4444'])
    ax.scatter(X_s[y_s==0, 0], X_s[y_s==0, 1], c='#3b82f6', s=15, alpha=0.5, label='Maioria')
    ax.scatter(X_s[y_s==1, 0], X_s[y_s==1, 1], c='#ef4444', s=30, alpha=0.8, label='Minoria')
    ax.set_title(titulo, fontsize=12)
    ax.legend(fontsize=8)

plt.suptitle("Deslocamento da fronteira de decisão do KNN com o aumento do desequilíbrio", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "fronteira_decisao_2d.png", dpi=150, bbox_inches="tight")
plt.show()

print("À medida que o IR aumenta, a fronteira de decisão desloca-se para cima da região")
print("da minoria, reduzindo a capacidade do KNN de detetar exemplos minoritários.")

In [ ]:
# Degradação do desempenho do KNNOptK com o IR nos datasets reais
if _optk_cache.exists():
    per_ds = opt_k_df.groupby("dataset")[["IR", "f1", "geometric_mean", "roc_auc"]].mean().reset_index()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metricas = [("f1", "F1-Score"), ("geometric_mean", "G-Mean"), ("roc_auc", "ROC AUC")]
    
    for ax, (metric, label) in zip(axes, metricas):
        ax.scatter(per_ds["IR"], per_ds[metric], alpha=0.6, edgecolors="k", s=40)
        slope, intercept, r_val, p_val, _ = sp_stats.linregress(per_ds["IR"], per_ds[metric])
        x_line = np.linspace(per_ds["IR"].min(), per_ds["IR"].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, 'r--', alpha=0.7,
                label=f'r={r_val:.2f}, p={p_val:.3f}')
        ax.set(xlabel="IR", ylabel=label, title=f"{label} vs. IR (KNNOptK)")
        ax.legend(fontsize=8)
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / "optk_degradacao_ir.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Correlação negativa clara: o desempenho degrada-se sistematicamente com o IR,")
    print("mesmo com k selecionado por cross-validation.")

---
<a id='5'></a>
## 5. Proposta: KNNFairRank — Correção por Estatísticas de Ordem

### 5.1 Motivação

A análise anterior mostra que o KNN falha com desequilíbrio de classes porque a **comparação de distâncias é injusta**: o vizinho rank-1 da maioria está sistematicamente mais próximo do que o vizinho rank-1 da minoria, simplesmente porque existem mais amostras maioritárias. A solução natural é **corrigir esta comparação**.

### 5.2 Derivação Teórica

Consideremos a distância do ponto de teste $x$ ao $k$-ésimo vizinho mais próximo da classe $c$, denotada $d_k^c(x)$. Assumindo um processo de Poisson uniforme com $N_c$ pontos da classe $c$ num espaço de dimensão $d$, a distância esperada é:

$$\mathbb{E}[d_k^c] \propto \left(\frac{k}{N_c}\right)^{1/d}$$

Para uma **comparação justa**, queremos encontrar o rank $k_{\text{eff}}$ na maioria tal que a distância esperada iguale a do rank 1 na minoria:

$$\mathbb{E}[d_{k_{\text{eff}}}^{\text{maj}}] = \mathbb{E}[d_1^{\text{min}}]$$

Substituindo:

$$\left(\frac{k_{\text{eff}}}{N_{\text{maj}}}\right)^{1/d} = \left(\frac{1}{N_{\text{min}}}\right)^{1/d}$$

Elevando ambos os lados à potência $d$ e simplificando:

$$\boxed{k_{\text{eff}} = r = \frac{N_{\text{maj}}}{N_{\text{min}}}}$$

**A dimensão $d$ cancela-se** — o rank efetivo é simplesmente o rácio de desequilíbrio global, sem necessidade de estimar a dimensionalidade intrínseca.

### 5.3 Algoritmo de Votação

O KNNFairRank usa **duas listas de vizinhos separadas por classe** e faz $n_{\text{votes}}$ comparações corrigidas por rank:

$$\text{Para } i = 1, \ldots, n_{\text{votes}}: \quad \text{voto}_i = \begin{cases} \text{minoria} & \text{se } d_i^{\text{min}} < d_{i \cdot k_{\text{eff}}}^{\text{maj}} \\ \text{maioria} & \text{caso contrário} \end{cases}$$

A previsão final é a classe com mais votos.

### 5.4 Variantes Propostas

| Variante | Descrição | Motivação |
|---|---|---|
| **KNNFairRank** (v3) | Votação binária com $k_{\text{eff}} = r$ | Derivação teórica direta |
| **KNNFairRankMagnitude** | Score contínuo $s_i = d_{\text{maj}} / (d_{\text{min}} + d_{\text{maj}})$ | Preserva informação da magnitude |
| **KNNFairRankCV** | $k_{\text{eff}} = \lfloor r^\alpha \rceil$ com $\alpha$ selecionado por inner CV | Adapta a força da correção ao dataset |

In [ ]:
# Demonstração visual: KNN padrão vs KNNFairRank num dataset sintético 2D
from src.algorithms.knn_base import KNNClassifierFast
from src.algorithms.knn_fair_rank import KNNFairRank

X_demo, y_demo = criar_dataset_2d(n_maj=200, n_min=20, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (clf, nome) in zip(axes, [(KNNClassifierFast(k=5), 'KNN Padrão (k=5)'),
                                    (KNNFairRank(), 'KNNFairRank')]):
    clf.fit(X_demo, y_demo)
    xx, yy = np.meshgrid(np.linspace(-3, 4, 150), np.linspace(-3, 4, 150))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = clf.predict(grid).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.15, levels=[0, 0.5, 1], colors=['#3b82f6', '#ef4444'])
    ax.scatter(X_demo[y_demo==0, 0], X_demo[y_demo==0, 1], c='#3b82f6', s=15, alpha=0.5, label='Maioria')
    ax.scatter(X_demo[y_demo==1, 0], X_demo[y_demo==1, 1], c='#ef4444', s=30, alpha=0.8, label='Minoria')
    ax.set_title(nome, fontsize=12)
    ax.legend(fontsize=8)

plt.suptitle("Comparação de fronteiras de decisão (IR = 10:1)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "fairrank_vs_padrao_2d.png", dpi=150, bbox_inches="tight")
plt.show()

print("O KNNFairRank expande a região da minoria ao corrigir o viés de comparação de ranks.")

---
<a id='6'></a>
## 6. Estudo Empírico — Configuração Experimental

### 6.1 Metodologia

| Parâmetro | Valor |
|---|---|
| Validação cruzada | 10-fold estratificada × 5 repetições |
| Escalonamento | StandardScaler (ajustado **apenas** no treino de cada fold) |
| Binarização de labels | Classe minoritária → 1 (positiva) |
| Features constantes | Removidas antes da avaliação |
| Semente aleatória | 42 (fixa para reprodutibilidade) |

### 6.2 Algoritmos Comparados

| Algoritmo | Tipo | Descrição |
|---|---|---|
| **KNNOptK** | Baseline | KNN com $k$ selecionado por inner CV (balanced accuracy) |
| **SMOTE+KNN** | Baseline externo | SMOTE (*imbalanced-learn*) + KNN |
| **KNNWeighted** | Baseline simples | KNN com pesos de classe balanceados |
| **KNNAdaptiveTopo** | Proposta exploratória | k adaptativo via homologia persistente |
| **KNNFairRank** | **Proposta principal** | Correção por rank com $k_{\text{eff}} = r$ |
| **KNNFairRankMagnitude** | Variante | Score contínuo (magnitude-aware) |
| **KNNFairRankCV** | **Variante principal** | $k_{\text{eff}} = r^\alpha$ com $\alpha$ tuned por inner CV |

### 6.3 Métricas

| Métrica | Fórmula | Papel |
|---|---|---|
| **MCC** | $\frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$ | Métrica primária — robusta para desequilíbrio |
| **G-Mean** | $\sqrt{\text{sensibilidade} \times \text{especificidade}}$ | Equilíbrio entre as classes |
| **F1-Score** | $\frac{2 \cdot \text{precisão} \times \text{recall}}{\text{precisão} + \text{recall}}$ | Desempenho na classe positiva (minoria) |
| **ROC AUC** | Área sob a curva ROC | Capacidade discriminativa global |

O benchmark guarda as contagens da matriz de confusão (TP, TN, FP, FN) em bruto — as métricas são derivadas a posteriori, permitindo adicionar novas métricas sem re-executar o benchmark.

In [ ]:
# ── Carregar ou executar o benchmark ──────────────────────────────────────────
import os
from imblearn.over_sampling import SMOTE
from src.evaluation.benchmarking import run_benchmark
from src.algorithms import (
    KNNOptK, KNNFairRank, KNNFairRankMagnitude, KNNFairRankCV,
    KNNWeighted, KNNAdaptiveTopo
)

# Wrapper SMOTE+KNN
class _SMOTEKNN:
    def __init__(self):
        smote_cfg = cfg.get("smote", {})
        self._smote = SMOTE(
            k_neighbors=smote_cfg.get("k_neighbors", 5),
            random_state=SEED,
        )
        self._clf = KNNClassifierFast(k=5)
        self.classes_ = None

    def fit(self, X, y):
        X_res, y_res = self._smote.fit_resample(X, y)
        self._clf.fit(X_res, y_res)
        self.classes_ = self._clf.classes_
        return self

    def predict(self, X):
        return self._clf.predict(X)

    def predict_proba(self, X):
        return self._clf.predict_proba(X)

# Algoritmos para benchmarking
ESTIMATORS = {
    "KNNOptK":             KNNOptK,
    "SMOTE+KNN":           _SMOTEKNN,
    "KNNWeighted":         KNNWeighted,
    "KNNAdaptiveTopo":     KNNAdaptiveTopo,
    "KNNFairRank":         KNNFairRank,
    "KNNFairRankMagnitude": KNNFairRankMagnitude,
    "KNNFairRankCV":       KNNFairRankCV,
}

N_REPS = 5  # 5 repetições para a entrega final
CACHE = TAB_DIR / f"benchmark_{N_REPS}rep.csv"

# Limitar threads para evitar sobrecarga
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

print(f"Ficheiro de resultados: {CACHE.name}")
print(f"Algoritmos: {list(ESTIMATORS.keys())}")
print(f"Datasets: {len(datasets)}")
print(f"Repetições: {N_REPS}")

In [ ]:
# ── Executar benchmark (com resume automático) ────────────────────────────────
bench_df = run_benchmark(
    estimators=ESTIMATORS,
    datasets=datasets,
    output_path=CACHE,
    n_jobs=1,
    n_repetitions=N_REPS,
)
print(f"\nResultados: {len(bench_df)} linhas, {bench_df['algorithm'].nunique()} algoritmos, "
      f"{bench_df['dataset'].nunique()} datasets.")

In [ ]:
# ── Derivar métricas a partir das contagens da matriz de confusão ─────────────
_CM_COLS = {"tp", "tn", "fp", "fn"}
if _CM_COLS.issubset(bench_df.columns):
    tp = bench_df["tp"]; tn = bench_df["tn"]
    fp = bench_df["fp"]; fn = bench_df["fn"]
    
    # F1
    prec = tp / (tp + fp).replace(0, np.nan)
    rec  = tp / (tp + fn).replace(0, np.nan)
    bench_df["f1"] = (2 * prec * rec / (prec + rec)).fillna(0)
    
    # Balanced Accuracy
    sens = tp / (tp + fn).replace(0, np.nan)
    spec = tn / (tn + fp).replace(0, np.nan)
    bench_df["balanced_accuracy"] = ((sens.fillna(0) + spec.fillna(0)) / 2)
    
    # G-Mean
    bench_df["geometric_mean"] = np.sqrt(sens.fillna(0) * spec.fillna(0))
    
    # MCC
    mcc_num = tp * tn - fp * fn
    mcc_den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    bench_df["mcc"] = np.where(mcc_den > 0, mcc_num / mcc_den, 0.0)
    
    # Precision e Recall separados
    bench_df["precision"] = prec.fillna(0)
    bench_df["recall"] = rec.fillna(0)
    
    print("Métricas derivadas com sucesso: f1, balanced_accuracy, geometric_mean, mcc, precision, recall")
else:
    print("Métricas já presentes no CSV (schema antigo).")

# Filtrar datasets degenerados
min_minority = 2 * cfg["evaluation"]["cv_folds"]
valid_datasets = [
    ds.name for ds in datasets
    if int(binarise_labels(ds.y).sum()) >= min_minority
]
removed = bench_df[~bench_df["dataset"].isin(valid_datasets)]
if len(removed) > 0:
    removed_names = removed["dataset"].unique()
    print(f"\nDatasets removidos (minoria < {min_minority}): {list(removed_names)}")
    bench_df = bench_df[bench_df["dataset"].isin(valid_datasets)].copy()

print(f"\nDatasets finais: {bench_df['dataset'].nunique()}")
print(f"Linhas: {len(bench_df)}")

---
<a id='7'></a>
## 7. Resultados e Análise

### 7.1 Desempenho Médio por Algoritmo

In [ ]:
from src.evaluation.statistical_tests import average_ranks, critical_difference

METRICS = ["mcc", "f1", "geometric_mean", "roc_auc"]
METRIC_LABELS = {"mcc": "MCC", "f1": "F1-Score", "geometric_mean": "G-Mean", "roc_auc": "ROC AUC"}

summary_table = (
    bench_df
    .groupby("algorithm")[METRICS]
    .mean()
    .round(4)
    .sort_values("mcc", ascending=False)
)

summary_table.to_csv(TAB_DIR / f"benchmark_summary_{N_REPS}rep.csv")

display(Markdown("### Tabela de Desempenho Médio (todos os datasets × todos os folds)"))
display(summary_table.style.highlight_max(axis=0, color='#a3e4a3').format(precision=4))

# Ordem para gráficos
ALG_ORDER = summary_table.index.tolist()

### 7.2 Boxplots — Distribuição por Fold

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, metric in zip(axes.ravel(), METRICS):
    data_plot = [bench_df[bench_df["algorithm"] == alg][metric].dropna().values for alg in ALG_ORDER]
    bp = ax.boxplot(data_plot, patch_artist=True, notch=False, widths=0.6)
    
    colors = sns.color_palette("husl", len(ALG_ORDER))
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_xticklabels(ALG_ORDER, rotation=35, ha="right", fontsize=9)
    ax.set_ylabel(METRIC_LABELS[metric])
    ax.set_title(f"Distribuição de {METRIC_LABELS[metric]}")
    ax.grid(axis='y', alpha=0.3)

plt.suptitle("Distribuição de métricas por algoritmo (todos os folds)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / f"boxplots_{N_REPS}rep.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.3 Scatter Per-Dataset vs. KNNOptK

In [ ]:
per_ds_alg = (
    bench_df
    .groupby(["dataset", "algorithm"])["mcc"]
    .mean()
    .unstack("algorithm")
)

competitors = [a for a in ALG_ORDER if a != "KNNOptK"]
n_cols = 3
n_rows = int(np.ceil(len(competitors) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
axes_flat = axes.ravel() if hasattr(axes, 'ravel') else [axes]

for ax, alg in zip(axes_flat, competitors):
    if alg not in per_ds_alg.columns or "KNNOptK" not in per_ds_alg.columns:
        continue
    x = per_ds_alg["KNNOptK"]
    y = per_ds_alg[alg]
    valid = x.notna() & y.notna()
    ax.scatter(x[valid], y[valid], alpha=0.5, s=30, edgecolors='k', linewidths=0.3)
    lims = [min(x[valid].min(), y[valid].min()) - 0.05, max(x[valid].max(), y[valid].max()) + 0.05]
    ax.plot(lims, lims, 'k--', alpha=0.3, label='Diagonal')
    n_wins = (y[valid] > x[valid]).sum()
    n_total = valid.sum()
    ax.set(xlabel="KNNOptK (MCC)", ylabel=f"{alg} (MCC)",
           title=f"{alg}  ({n_wins}/{n_total} vitórias)")

# Esconder eixos sobrantes
for i in range(len(competitors), len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.suptitle("MCC per-dataset: cada algoritmo vs. KNNOptK (acima da diagonal = melhoria)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / f"scatter_vs_optk_{N_REPS}rep.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.4 Desempenho por Severidade do Desequilíbrio (Quartis de IR)

In [ ]:
bench_df["IR_quartile"] = pd.qcut(bench_df["imbalance_ratio"], q=4,
                                   labels=["Q1 (severo)", "Q2", "Q3", "Q4 (leve)"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, metric, label in zip(axes, ["mcc", "geometric_mean"], ["MCC", "G-Mean"]):
    ir_summary = (
        bench_df.groupby(["IR_quartile", "algorithm"])[metric]
        .mean().unstack("algorithm").round(4)
    )
    ir_summary[ALG_ORDER].plot(kind="bar", ax=ax, width=0.8)
    ax.set(xlabel="Quartil de IR", ylabel=label,
           title=f"{label} por quartil de severidade do desequilíbrio")
    ax.legend(fontsize=7, ncol=2)
    ax.tick_params(axis='x', rotation=0)
    
    ir_summary.to_csv(TAB_DIR / f"benchmark_by_ir_quartile_{metric}_{N_REPS}rep.csv")

plt.tight_layout()
plt.savefig(FIG_DIR / f"ir_quartile_{N_REPS}rep.png", dpi=150, bbox_inches="tight")
plt.show()

print("Nota: Em Q1 (desequilíbrio severo), as variantes FairRank destacam-se em G-Mean")
print("porque a correção de rank é mais impactante quando o IR é elevado.")

### 7.5 Análise Per-Classe: Recall da Minoria vs. Maioria

In [ ]:
# Calcular métricas per-classe a partir das contagens
if _CM_COLS.issubset(bench_df.columns):
    per_class = bench_df.groupby("algorithm").agg(
        tp=("tp", "mean"), tn=("tn", "mean"),
        fp=("fp", "mean"), fn=("fn", "mean")
    )
    
    # Recall minoria = TP / (TP + FN)
    per_class["recall_minoria"] = per_class["tp"] / (per_class["tp"] + per_class["fn"]).replace(0, np.nan)
    # Recall maioria = TN / (TN + FP)
    per_class["recall_maioria"] = per_class["tn"] / (per_class["tn"] + per_class["fp"]).replace(0, np.nan)
    # Precision minoria
    per_class["precision_minoria"] = per_class["tp"] / (per_class["tp"] + per_class["fp"]).replace(0, np.nan)
    # F1 minoria
    prec_m = per_class["precision_minoria"]
    rec_m = per_class["recall_minoria"]
    per_class["f1_minoria"] = 2 * prec_m * rec_m / (prec_m + rec_m)
    # Gap
    per_class["gap_recall"] = per_class["recall_minoria"] - per_class["recall_maioria"]
    
    per_class_display = per_class[["recall_minoria", "precision_minoria", "f1_minoria",
                                   "recall_maioria", "gap_recall"]].round(4)
    per_class_display = per_class_display.sort_values("gap_recall", ascending=False)
    per_class_display.to_csv(TAB_DIR / f"per_class_summary_{N_REPS}rep.csv")
    
    display(Markdown("### Desempenho por Classe"))
    display(Markdown(
        "O **gap de recall** mede a diferença entre recall da minoria e da maioria. "
        "Valores perto de 0 indicam equilíbrio entre as classes."
    ))
    display(per_class_display.style.format(precision=4)
            .background_gradient(subset=["recall_minoria"], cmap="Greens")
            .background_gradient(subset=["gap_recall"], cmap="RdYlGn"))

    # Gráfico do trade-off recall minoria vs recall maioria
    fig, ax = plt.subplots(figsize=(8, 6))
    for alg in per_class_display.index:
        ax.scatter(per_class_display.loc[alg, "recall_maioria"],
                   per_class_display.loc[alg, "recall_minoria"],
                   s=100, zorder=5)
        ax.annotate(alg, (per_class_display.loc[alg, "recall_maioria"],
                           per_class_display.loc[alg, "recall_minoria"]),
                    textcoords="offset points", xytext=(5, 5), fontsize=8)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Equilíbrio perfeito')
    ax.set(xlabel="Recall Maioria", ylabel="Recall Minoria",
           title="Trade-off: Recall Minoria vs. Recall Maioria")
    ax.legend()
    ax.set_xlim(0.6, 1.0)
    ax.set_ylim(0.3, 1.0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"tradeoff_recall_{N_REPS}rep.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Contagens TP/TN/FP/FN não disponíveis — a usar schema antigo.")

### 7.6 Intervalos de Confiança por Bootstrap

In [ ]:
B_BOOT = 2000
ALPHA_CI = 0.05
rng_ci = np.random.default_rng(SEED)

per_ds_metric = bench_df.groupby(["dataset", "algorithm"])[METRICS].mean()

ci_rows = []
for metric in METRICS:
    pivot = per_ds_metric[metric].unstack("algorithm")
    for alg in pivot.columns:
        vals = pivot[alg].dropna().values
        boot_means = np.array([
            rng_ci.choice(vals, size=len(vals), replace=True).mean()
            for _ in range(B_BOOT)
        ])
        ci_rows.append({
            "algorithm": alg, "metric": metric,
            "mean": round(vals.mean(), 4),
            "ci_lower": round(np.percentile(boot_means, 100 * ALPHA_CI / 2), 4),
            "ci_upper": round(np.percentile(boot_means, 100 * (1 - ALPHA_CI / 2)), 4),
            "ci_width": round(np.percentile(boot_means, 97.5) - np.percentile(boot_means, 2.5), 4),
            "n_datasets": len(vals),
        })

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(TAB_DIR / f"bootstrap_ci_{N_REPS}rep.csv", index=False)

# Gráfico de intervalos de confiança
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, metric in zip(axes.ravel(), METRICS):
    m_ci = (ci_df[ci_df["metric"] == metric]
            .set_index("algorithm")
            .sort_values("mean", ascending=False))
    algs = m_ci.index.tolist()
    y_pos = np.arange(len(algs))
    
    colors = sns.color_palette("husl", len(algs))
    ax.barh(y_pos, m_ci["mean"], xerr=[m_ci["mean"] - m_ci["ci_lower"],
                                        m_ci["ci_upper"] - m_ci["mean"]],
            color=colors, alpha=0.7, capsize=3, edgecolor='grey', linewidth=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(algs, fontsize=9)
    ax.set_xlabel(METRIC_LABELS[metric])
    ax.set_title(f"{METRIC_LABELS[metric]} — IC 95% Bootstrap")
    ax.invert_yaxis()

plt.suptitle("Desempenho médio com intervalos de confiança (bootstrap, 2000 amostras)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / f"bootstrap_ci_{N_REPS}rep.png", dpi=150, bbox_inches="tight")
plt.show()

---
<a id='8'></a>
## 8. Testes Estatísticos

Seguimos o protocolo de Demšar (2006):
1. **Teste de Friedman** — hipótese nula: todos os algoritmos têm desempenho igual
2. **Testes de Wilcoxon post-hoc** com correção de Holm — comparações par a par contra uma baseline
3. **Diagrama de Diferença Crítica** — visualização dos ranks médios

In [ ]:
from src.evaluation.statistical_tests import friedman_test, pairwise_wilcoxon, average_ranks, critical_difference

# ── Friedman test ─────────────────────────────────────────────────────────────
for metric in ["mcc", "geometric_mean"]:
    try:
        stat, p = friedman_test(bench_df, metric=metric)
        print(f"Friedman ({METRIC_LABELS[metric]}): χ² = {stat:.2f}, p = {p:.2e}"
              f" {'→ Rejeitar H₀ ✓' if p < 0.05 else '→ Não rejeitar H₀'}")
    except Exception as e:
        print(f"Friedman ({metric}): erro — {e}")

In [ ]:
# ── Wilcoxon post-hoc vs. baselines ───────────────────────────────────────────

def relatorio_wilcoxon(bench_df, baseline, metric, label):
    """Executa e mostra o teste de Wilcoxon pairwise vs. uma baseline."""
    wilcox = pairwise_wilcoxon(bench_df, baseline=baseline, metric=metric)
    safe_base = baseline.replace("+", "").replace(" ", "_")
    wilcox.to_csv(TAB_DIR / f"wilcoxon_vs_{safe_base}_{metric}_{N_REPS}rep.csv", index=False)
    
    display(Markdown(f"#### Wilcoxon vs. **{baseline}** ({label})"))
    styled = wilcox.style.format({
        "statistic": "{:.0f}", "p_raw": "{:.2e}",
        "p_corrected": "{:.4f}", "significant": "{}"
    }).map(lambda v: "color: green; font-weight: bold" if v is True else "",
                subset=["significant"])
    display(styled)
    return wilcox

# Comparações principais
for baseline in ["KNNOptK", "SMOTE+KNN", "KNNFairRank"]:
    for metric, label in [("mcc", "MCC"), ("geometric_mean", "G-Mean")]:
        try:
            relatorio_wilcoxon(bench_df, baseline, metric, label)
        except Exception as e:
            print(f"Wilcoxon {baseline}/{metric}: {e}")

In [ ]:
# ── Diagrama de Diferença Crítica ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

for ax, metric, label in zip(axes, ["mcc", "geometric_mean"], ["MCC", "G-Mean"]):
    ranks = average_ranks(bench_df, metric=metric)
    cd = critical_difference(bench_df, metric=metric)
    
    ranks.to_csv(TAB_DIR / f"average_ranks_{metric}_{N_REPS}rep.csv")
    
    print(f"\nRanks Médios ({label}):")
    for alg, rank in ranks.items():
        print(f"  {alg:25s}  {rank:.3f}")
    print(f"  Diferença Crítica (Nemenyi, α=0.05): {cd:.3f}")
    
    # Diagrama
    names = ranks.index.tolist()
    vals = ranks.values
    
    ax.scatter(vals, np.zeros_like(vals), zorder=3, s=60)
    for name, v in zip(names, vals):
        ax.annotate(name, (v, 0), textcoords="offset points",
                    xytext=(0, 12), ha="center", fontsize=7, rotation=30)
    
    best = vals.min()
    ax.axhline(0, color="black", linewidth=0.8)
    ax.hlines(0, best, best + cd, colors="red", linewidths=3, label=f"CD = {cd:.3f}")
    ax.set_yticks([])
    ax.set_xlabel("Rank Médio (menor = melhor)")
    ax.set_title(f"Diagrama de Diferença Crítica — {label}")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / f"cd_diagram_{N_REPS}rep.png", dpi=150, bbox_inches="tight")
plt.show()

---
<a id='9'></a>
## 9. Discussão

### 9.1 Síntese dos Resultados

Os resultados mostram um padrão claro:

- **Em MCC e F1**, o KNNOptK e o KNNFairRankCV são os melhores algoritmos, seguidos pelo SMOTE+KNN. As variantes FairRank base e Magnitude ficam atrás porque sacrificam precisão da minoria para ganhar recall.

- **Em G-Mean e ROC AUC**, o **KNNFairRankCV é o melhor algoritmo**, superando todos os baselines incluindo o SMOTE+KNN. As variantes FairRank dominam porque são desenhadas para equilibrar sensibilidade e especificidade.

- O **KNNAdaptiveTopo** obtém resultados intermédios — a abordagem baseada em homologia persistente é interessante mas não captura o viés estrutural tão diretamente como o FairRank.

### 9.2 O Trade-off Recall–Precisão

A análise per-classe (Secção 7.5) revela o mecanismo do trade-off:

| Algoritmo | Recall Minoria | Precisão Minoria | Interpretação |
|---|---|---|---|
| KNNOptK | ~0.54 | ~0.59 | Conservador: quase não deteta a minoria |
| KNNFairRankCV | ~0.84 | ~0.52 | Equilibrado: deteta a maioria da minoria |
| KNNFairRankMagnitude | ~0.88 | ~0.42 | Agressivo: altíssimo recall, baixa precisão |

O KNNFairRankCV (com $\alpha$ tuned por CV) encontra o melhor ponto de operação: faz correção suficiente para melhorar drasticamente o recall da minoria, mas o $\alpha < 1$ evita sobre-correção.

### 9.3 Abordagens que Não Funcionaram

Durante o desenvolvimento, explorámos várias direções que não resultaram tão bem:

- **DANN (Discriminant Adaptive NN)**: A adaptação da métrica local de Hastie & Tibshirani não resolve o viés de votação — muda a geometria mas não corrige a contagem desigual de vizinhos.

- **Adaptive-k por Entropia/Eigenvalores**: Escolher k por critérios geométricos locais não aborda diretamente o problema do desequilíbrio — o k ótimo localmente não compensa o viés de amostragem.

- **FairRank com rácio local de desequilíbrio**: Substituir $r$ global por $r_{\text{local}}$ introduz ruído sem ganho — o rácio local é mal estimado com poucos vizinhos minoritários.

### 9.4 Limitações

1. A derivação do KNNFairRank assume uma distribuição de Poisson uniforme, que é violada em dados reais (clusters, distribuições não uniformes).
2. O KNNFairRankCV resolve parcialmente esta limitação com o parâmetro $\alpha$, mas $\alpha$ é global — não se adapta a regiões diferentes do espaço.
3. Alguns datasets da suite são originalmente multiclass e foram binarizados, o que pode afetar a distribuição de classes.
4. O tempo de execução do KNNFairRank é superior ao do KNN padrão devido ao cálculo de distâncias separadas por classe.

---
<a id='10'></a>
## 10. Conclusões e Trabalho Futuro

### 10.1 Conclusões

1. **O KNN padrão falha sistematicamente com desequilíbrio de classes**, mesmo com seleção ótima de $k$ — o problema é estrutural, não paramétrico.

2. **O KNNFairRank resolve o viés pela raiz**: ao corrigir a comparação de ranks usando a igualdade $k_{\text{eff}} = r$ derivada de estatísticas de ordem, elimina o viés de amostragem sem alterar os dados de treino.

3. **O KNNFairRankCV é a variante mais robusta**: o parâmetro $\alpha$ (tuned por inner CV) permite adaptar a força da correção a cada dataset, obtendo o melhor equilíbrio entre sensibilidade e especificidade.

4. **Resultados competitivos**: O KNNFairRankCV é #1 em G-Mean e ROC AUC (superando o SMOTE+KNN) e top-2 em MCC, demonstrando que a correção algorítmica é uma alternativa viável ao *oversampling*.

### 10.2 Trabalho Futuro

- **Correção local**: Substituir $r$ global por uma estimativa de densidade local $r(x) = \hat{\lambda}_{\text{maj}}(x) / \hat{\lambda}_{\text{min}}(x)$ usando estimadores *k*-NN de densidade.
- **Fallback por confiança**: Quando o score de votação está próximo de 0.5 (incerto), delegar a decisão a um classificador conservador.
- **Extensão a multiclass**: Generalizar a correção de rank para problemas com mais de duas classes.
- **Combinação com TDA**: Usar homologia persistente para identificar regiões de fronteira e aplicar correções diferentes.

---
## Referências

1. rushter/MLAlgorithms — https://github.com/rushter/MLAlgorithms/blob/master/mla/knn.py
2. Hastie, T. & Tibshirani, R. (1996). *Discriminant Adaptive Nearest Neighbor Classification*.
3. Demšar, J. (2006). *Statistical Comparisons of Classifiers over Multiple Data Sets*. JMLR.
4. Chawla, N.V. et al. (2002). *SMOTE: Synthetic Minority Over-sampling Technique*.
5. Levina, E. & Bickel, P.J. (2005). *Maximum Likelihood Estimation of Intrinsic Dimension*.